# 2. Train Baseline Fraud Detection Model

Build a fraud detector using XGBoost and save it for attacks.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import pickle
import sys
sys.path.insert(0, '../src')
from utils import evaluate_model

# Load data
df = pd.read_csv('../data/creditcard.csv')
X = df.drop('Class', axis=1).values
y = df['Class'].values

print(f"Dataset shape: {X.shape}")
print(f"Fraud samples: {y.sum()}")

Dataset shape: (284807, 30)
Fraud samples: 492


## Preprocess & Split Data

In [2]:
# Normalize features to [0,1] range for attacks
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Split: train/val/test = 60/20/20
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y, test_size=0.4, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape}, Fraud: {y_train.sum()}")
print(f"Val: {X_val.shape}, Fraud: {y_val.sum()}")
print(f"Test: {X_test.shape}, Fraud: {y_test.sum()}")

Train: (170884, 30), Fraud: 295
Val: (56961, 30), Fraud: 98
Test: (56962, 30), Fraud: 99


## Train XGBoost Model

In [3]:
# Handle class imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale pos weight: {scale_pos_weight}")

# Train model with regularization for robustness
model = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    max_depth=4,              # Reduce depth (less overfitting)
    learning_rate=0.05,       # Lower learning rate (more conservative)
    n_estimators=200,         # More trees but weaker individually
    subsample=0.8,            # Subsample rows (regularization)
    colsample_bytree=0.8,     # Subsample features (regularization)
    reg_alpha=1.0,            # L1 regularization
    reg_lambda=1.0,           # L2 regularization
    random_state=42,
    eval_metric='auc',
    verbosity=1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("\nModel trained!")

Scale pos weight: 578.2677966101695



Model trained!


## Evaluate Model

In [4]:
# Predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Metrics
metrics = evaluate_model(y_test, y_pred, y_pred_proba)
print(f"Test Set Performance:")
for metric, value in metrics.items():
    print(f"  {metric}: {value:.4f}")

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred))

Test Set Performance:
  accuracy: 0.9985
  precision: 0.5506
  recall: 0.8788
  f1: 0.6770
  auc: 0.9723

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56863
           1       0.55      0.88      0.68        99

    accuracy                           1.00     56962
   macro avg       0.78      0.94      0.84     56962
weighted avg       1.00      1.00      1.00     56962



## Save Model & Scaler

In [5]:
# Save model
with open('../data/model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save scaler
with open('../data/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save feature names
feature_names = df.drop('Class', axis=1).columns.tolist()
with open('../data/feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

print("Model, scaler, and feature names saved!")

Model, scaler, and feature names saved!
